Need to make scaling method work with subset, or pass it the tensor of images instead, or could just scale once since all methods other than normalize are just functions.

subset[0] is one gal -> (img, flag)

In [ ]:
import gc
import sys
from pathlib import Path
import copy

import numpy as np
import pandas as pd
import torch
from astropy.io import fits
from sklearn.model_selection import KFold
from torch import nn
from torch.utils.data import Subset
from torchvision import transforms
from torchvision.transforms import Normalize

In [26]:
cwd = Path.cwd()
home_dir = cwd.parent.parent
cutout_dir = home_dir / 'data' / 'cutouts'
log_dir = home_dir / 'logs'
model_dir = cwd / 'models'

In [3]:
sys.path.append(str(home_dir))
import utils as ut

In [23]:
logdf = pd.read_json(log_dir / 'conv.json')
logdf = logdf.sort_values(by='accuracy', ascending=False).head()

params = logdf.iloc[0].to_dict()
params

{'n_conv': 2,
 'n_fc': 1,
 'n_channels_conv0': 23,
 'n_channels_conv1': 25,
 'n_channels_conv2': 18,
 'kernel_size_conv0': 3,
 'kernel_size_conv1': 2,
 'kernel_size_conv2': 2,
 'padding_conv0': 2,
 'padding_conv1': 1,
 'padding_conv2': 1,
 'dropout_conv0': 0.20942875570602001,
 'dropout_conv1': 0.39092312337913404,
 'dropout_conv2': 0.294306794322898,
 'n_units_fc0': 132,
 'n_units_fc1': 236,
 'n_units_fc2': 64,
 'dropout_fc0': 0.323114876910688,
 'dropout_fc1': 0.426665341562914,
 'dropout_fc2': 0.26863944964748604,
 'lr': 2.031980983842495e-05,
 'optimizer': 'AdamW',
 'w1': 2.266807513020847,
 'w2': 2.742921180375435,
 'batch_size': 64,
 'comment': 'test k fold validation',
 'bands': ['f150w', 'f277w', 'f444w'],
 'out_features': ['FLAG'],
 'seed': 42,
 'epochs': 200,
 'image_size': 67,
 'num_trials': 20,
 'patience': 20,
 'accuracy': 0.895350062847137,
 'best accuracy': 1.0,
 'adamw_weight_decay': 0.017079750342958,
 'scaling': 'tanh',
 'sgd_momentum': nan,
 'N_folds': 5.0}

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUT_FEATURES = ['FLAG']
SEED = params['seed']

# HYPERPARAMETERS
INPUT_SIZE = params['image_size']
EPOCHS = params['epochs']
NUM_TRIALS = params['num_trials']
PATIENCE = params['patience']
SCALER = params['scaling']
N_SPLITS = params['N_folds']
BANDS = params['bands']

In [7]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

if SEED:
    ut.set_seeds(SEED)

## Prepare Data

In [8]:
# LOAD DATA INTO DF

cols = {'idx': [], 'images': [], 'header': []}

for folder in cutout_dir.iterdir():
    if folder.is_dir():
        cols['idx'].append(folder.stem)

        temp_df = ut.fits_df(folder)
        temp_df.sort_values(by='BAND')

        # load images
        temp_imgs = {}
        for _, row in temp_df.iterrows():
            hdul = fits.open(row['FILEPATH'])
            data = hdul[0].data.astype('float32')
            header = hdul[0].header
            temp_imgs[row['BAND']] = data
            hdul.close()
        cols['images'].append(temp_imgs)
        cols['header'].append(header)

# turn dict into df and expand header columns
df = pd.DataFrame(cols)
to_expand = ['header', 'images']
for col in to_expand:
    temp_df = pd.DataFrame(df[col].tolist(), index=df.index)
    df = df.join(temp_df)

df.drop(columns=['header', 'BAND', 'images'], inplace=True)

# shuffle
df = df.sample(frac=1, random_state=SEED) if SEED else df.sample(frac=1)

In [10]:
# STACK AND RESIZE BANDS

resize_transform = transforms.Resize(
    INPUT_SIZE, transforms.InterpolationMode.BILINEAR
)

df['images'] = df[BANDS].values.tolist()
df['images'] = [resize_transform(torch.as_tensor(img)) for img in df['images']]

C:\Users\harty\AppData\Local\Temp\ipykernel_8524\3055811126.py:8: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  df['images'] = [resize_transform(torch.as_tensor(img)) for img in df['images']]


In [11]:
# EXTRACT X, y, and i from data

data_df = df[df['TABLE'] != 'test']

X_data = torch.stack(data_df['images'].tolist())
y_data = torch.tensor(data_df['FLAG'].tolist())
i_data = torch.tensor(data_df['idx'].astype(int).tolist())

In [12]:
X_data.shape

torch.Size([438, 3, 67, 67])

In [13]:
# SCALE DATA


def scale_data(data, type='tanh'):

    if type == 'log1p':
        data = torch.log1p(data)
    elif type == 'pytorch_normalize':
        scaler = Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
        data = scaler(data)
    elif type == 'arcsinh':
        data = torch.asinh(data)
    elif type == 'tanh':
        data = torch.tanh(data)
    else:
        raise ValueError(f'{type} is not a valid scaling function')

    return data


X_data_scaled = scale_data(X_data, type=SCALER)
ut.analyze_data(X_data, axis=(0, 2, 3), name='Unscaled data:')
print('\n')
ut.analyze_data(X_data_scaled, axis=(0, 2, 3), name=f'{SCALER} Scaled data:')

Unscaled data:
Mean:  [0.04243751 0.10102423 0.16987188]
Median:  [0.01113511 0.01979694 0.02893846]
St. Dev:  [0.17126837 0.3822131  0.7546609 ]
Max:  [17.483625 34.802776 80.58712 ]
Min:  [-0.06790778 -0.0484455  -0.17565338]
Length:  438


tanh Scaled data:
Mean:  [0.03739235 0.07838354 0.10648803]
Median:  [0.01113465 0.01979436 0.02893038]
St. Dev:  [0.09199782 0.1575378  0.2064333 ]
Max:  [1. 1. 1.]
Min:  [-0.06780359 -0.04840764 -0.17386886]
Length:  438


In [14]:
# CONVERT TO DATSET

# transforms to apply to inputs
train_transform = transforms.Compose(
    [
        # transforms.RandomHorizontalFlip(p=0.5),
        # transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(degrees=(0, 360))
        # transforms.RandomPerspective()
    ]
)

dataset = ut.GalaxyDataset(images=X_data_scaled, ids=i_data, labels=y_data)

In [15]:
# KFOLD TESTING

kfold = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
folds = list(kfold.split(dataset))

first_train_idx, first_test_idx = folds[0]

subset = Subset(dataset, first_train_idx)

## Define Model

In [ ]:
# DEFINE MODEL AND HYPERPARAMTER FUNCTIONS


def define_model(n_conv, n_fc, **kwargs):

    conv_channels = [
        params[f'n_channels_conv{layer}'] for layer in range(n_conv)
    ]
    conv_ksizes = [
        params[f'kernel_size_conv{layer}'] for layer in range(n_conv)
    ]
    conv_paddings = [params[f'padding_conv{layer}'] for layer in range(n_conv)]
    conv_dropouts = [params[f'dropout_conv{layer}'] for layer in range(n_conv)]

    fc_units = [params[f'n_units_fc{layer}'] for layer in range(n_fc)]
    fc_dropouts = [[params[f'dropout_fc{layer}'] for layer in range(n_fc)]]

    layers = []

    # build conv layers
    in_channels = len(BANDS)
    current_spatial_size = INPUT_SIZE

    for i in range(n_conv):
        out_channels = conv_channels[i]
        ksize = conv_ksizes[i]
        pad = conv_paddings[i]
        dropout = conv_dropouts[i]

        layers.extend(
            (
                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size=ksize,
                    padding=pad,
                    bias=False,
                ),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(),
                nn.Dropout(dropout),
            )
        )

        current_spatial_size = current_spatial_size - ksize + (2 * pad) + 1
        in_channels = out_channels

    flattened_features = in_channels * (current_spatial_size**2)
    layers.append(nn.Flatten())

    # build fc layers
    in_features = flattened_features
    for i in range(n_fc):
        out_features = fc_units[i]
        dropout = fc_dropouts[i]

        layers.extend(
            (
                nn.Linear(in_features, out_features, bias=False),
                nn.BatchNorm1d(out_features),
                nn.ReLU(),
                nn.Dropout(dropout),
            )
        )

        in_features = out_features

    layers.append(nn.Linear(in_features, 2 * len(OUT_FEATURES)))

    return nn.Sequential(*layers)


def get_hyperparameters(model, lr, optimizer, w1, w2, batch_size, **kwargs):

    # optimizer and learning rate
    optimizer_name = optimizer

    temp_kwargs = {'lr': lr}

    if optimizer_name == 'AdamW':
        temp_kwargs['weight_decay'] = kwargs['adamw_weight_decay']
    elif optimizer_name == 'SGD':
        temp_kwargs['momentum'] = kwargs['sgd_momentum']

    optimizer = getattr(torch.optim, optimizer_name)(
        model.parameters(), **temp_kwargs
    )

    # weights and loss function
    weights = torch.tensor([w1, w2]).to(DEVICE)
    loss_func = nn.CrossEntropyLoss(weight=weights)

    return optimizer, loss_func, batch_size

## Train Model

In [ ]:
model = define_model(**params).to(DEVICE)
optimizer, loss_func, batch_size = get_hyperparameters(model=model, **params)

kfold = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
all_folds_history = {}

# Split the dataset into 5 folds
for fold, (train_idx, val_idx) in enumerate(kfold.split(dataset)):
    history = {
        'train_loss': [],
        'val_loss': [],
        'train_acc': [],
        'val_acc': [],
    }
    # Create subsets for training and validation
    train_subset = Subset(dataset, train_idx)
    val_subset = Subset(dataset, val_idx)

    if SEED:
        train_loader = torch.utils.data.DataLoader(
            train_subset,
            batch_size=batch_size,
            shuffle=False,
            drop_last=True,
        )
    else:
        train_loader = torch.utils.data.DataLoader(
            train_subset,
            batch_size=batch_size,
            shuffle=True,
            drop_last=True,
        )
    val_loader = torch.utils.data.DataLoader(
        val_subset, batch_size=batch_size, shuffle=False
    )

    # early stopper
    stopper = ut.EarlyStopper(patience=PATIENCE, min_delta=0.0005)
    best_val_loss = float('inf')
    best_model_weights = None
    for _ in range(EPOCHS):
        # Training Loop ----------------------------------------------------
        train_loss, train_correct = ut.do_epoch(
            model=model,
            loader=train_loader,
            loss_function=loss_func,
            device=DEVICE,
            optimizer=optimizer,
            train=True,
        )

        # Validation Loop ---------------------------------------------------
        with torch.no_grad():
            val_loss, val_correct = ut.do_epoch(
                model=model,
                loader=val_loader,
                loss_function=loss_func,
                device=DEVICE,
                train=False,
            )

        epoch_val_loss = val_loss / len(val_subset)

        history['train_loss'].append(train_loss / len(train_subset))
        history['val_loss'].append(epoch_val_loss)
        history['train_acc'].append(train_correct / len(train_subset))
        history['val_acc'].append(val_correct / len(val_subset))

        # track loss once per epoch to check early stopper
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            best_model_weights = copy.deepcopy(model.state_dict())
        stopper(val_loss / len(val_subset))
        if stopper.early_stop:
            break

    model_path = model_dir / f'fold{fold}'
    torch.save(best_model_weights, model_path)
    all_folds_history[f'fold_{fold}'] = history

torch.save(all_folds_history, model_dir / 'loss_history.pt')